<a href="https://oscar-defelice.github.io">
<img src="../../img/image.png" height="125" align="right" />
</a>

# TP 01 — SOLUTION — Tokenisation, Text Normalisation & Datasets

> ⚠️ **Instructor / self-study use only.** Do not distribute before the session.

---

## Setup

In [ ]:
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score,
)

import nltk
import tiktoken
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
BPE_ENC = tiktoken.encoding_for_model("gpt-4")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
DATASET_NAME = "ag_news"

print("Setup OK.")

---
## Part 1 — Text Normalisation

In [ ]:
def preprocess_text(text: str) -> str:
    """
    Apply a minimal, reproducible normalisation pipeline to a single string.

    Steps (in order):
      1. Lowercase
      2. Remove HTML tags
      3. Replace non-alphanumeric, non-space characters with a space
      4. Collapse multiple spaces
      5. Strip

    Examples
    --------
    >>> preprocess_text("Hello,   World! <br/>")
    'hello world'
    """
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)       # remove HTML tags
    text = re.sub(r"[^a-z0-9 ]", " ", text)    # keep letters, digits, spaces
    text = re.sub(r" +", " ", text)             # collapse spaces
    return text.strip()


def preprocess_no_lowercase(text: str) -> str:
    """
    Same pipeline without step 1 (no lowercasing).
    """
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^A-Za-z0-9 ]", " ", text)
    text = re.sub(r" +", " ", text)
    return text.strip()


def preprocess_keep_punctuation(text: str) -> str:
    """
    Steps 1, 2, 4, 5 only — punctuation kept.
    """
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r" +", " ", text)
    return text.strip()


# Assertions
assert preprocess_text("Hello,   World! <br/>")        == "hello world"
assert preprocess_text("  NLP is FUN!!!  ")             == "nlp is fun"
assert preprocess_text("<p>U.S. stocks rose 2.3%</p>")  == "u s stocks rose 2 3"
assert preprocess_text("already clean")                == "already clean"
assert preprocess_text("")                              == ""
print("All assertions passed ✓")

---
## Part 2 — Tokenisation Comparison

In [ ]:
def tokenize_whitespace(text: str) -> list[str]:
    """
    Preprocess then split on whitespace.
    """
    return preprocess_text(text).split()


def tokenize_nltk(text: str) -> list[str]:
    """
    Preprocess then apply NLTK word_tokenize.
    """
    return nltk.word_tokenize(preprocess_text(text))


def tokenize_bpe(text: str) -> list[int]:
    """
    Apply GPT-4 BPE directly on raw text (no preprocessing).
    """
    return BPE_ENC.encode(text)


sample = "The Federal Reserve raised interest rates by 0.25%."
print("whitespace :", tokenize_whitespace(sample))
print("nltk       :", tokenize_nltk(sample))
print("bpe        :", tokenize_bpe(sample))

In [ ]:
def compare_tokenisers(texts: list[str], tokenisers: dict) -> pd.DataFrame:
    """
    Compute vocab_size, mean_length, median_length, oov_proxy for each tokeniser.

    oov_proxy = fraction of unique tokens that appear exactly once (hapax legomena).
    """
    rows = []
    for name, fn in tokenisers.items():
        all_tokens = []
        lengths    = []
        for t in texts:
            toks = fn(t)
            all_tokens.extend(toks)
            lengths.append(len(toks))

        counts    = Counter(all_tokens)
        vocab     = set(all_tokens)
        hapax     = sum(1 for c in counts.values() if c == 1)

        rows.append({
            "tokeniser"     : name,
            "vocab_size"    : len(vocab),
            "mean_length"   : round(float(np.mean(lengths)), 1),
            "median_length" : float(np.median(lengths)),
            "oov_proxy"     : round(hapax / len(vocab), 3) if vocab else 0,
        })
    return pd.DataFrame(rows).set_index("tokeniser")


sample_raw   = load_dataset(DATASET_NAME, split="train[:500]")
sample_texts = sample_raw["text"]

tokenisers = {
    "whitespace": tokenize_whitespace,
    "nltk"      : tokenize_nltk,
    "bpe"       : tokenize_bpe,
}

comparison = compare_tokenisers(sample_texts, tokenisers)
comparison

In [ ]:
def plot_vocab_growth(texts: list[str], tokenisers: dict, step: int = 20) -> None:
    """
    Plot cumulative vocabulary size vs number of texts seen.
    """
    fig, ax = plt.subplots(figsize=(8, 4))

    for name, fn in tokenisers.items():
        vocab   = set()
        x_vals  = []
        y_vals  = []
        for i, text in enumerate(texts):
            vocab.update(fn(text))
            if (i + 1) % step == 0 or i == len(texts) - 1:
                x_vals.append(i + 1)
                y_vals.append(len(vocab))
        ax.plot(x_vals, y_vals, label=name, marker="o", markersize=3)

    ax.set_xlabel("Number of texts")
    ax.set_ylabel("Vocabulary size")
    ax.set_title("Vocabulary growth by tokeniser")
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_vocab_growth(sample_texts, tokenisers, step=25)

---
## Part 3 — Dataset Inspection & Split

In [ ]:
def load_text_dataset(name: str) -> tuple[pd.DataFrame, dict]:
    """
    Load a HuggingFace text classification dataset into a unified DataFrame.
    """
    configs = {
        "ag_news" : {"label_names": ["World", "Sports", "Business", "Sci/Tech"], "text_col": "text"},
        "imdb"    : {"label_names": ["Negative", "Positive"],                   "text_col": "text"},
        "emotion" : {"label_names": ["sadness", "joy", "love", "anger", "fear", "surprise"], "text_col": "text"},
    }
    if name not in configs:
        raise ValueError(f"Unknown dataset '{name}'.")

    cfg = configs[name]
    raw = load_dataset(name if name != "emotion" else "dair-ai/emotion")

    frames = []
    for split_name, split_data in raw.items():
        df_split = split_data.to_pandas().rename(columns={cfg["text_col"]: "text"})
        df_split["split"]      = split_name
        df_split["label_name"] = df_split["label"].apply(lambda i: cfg["label_names"][i])
        frames.append(df_split[["text", "label", "label_name", "split"]])

    df   = pd.concat(frames, ignore_index=True)
    meta = {"label_names": cfg["label_names"], "num_classes": len(cfg["label_names"]), "total_examples": len(df)}
    return df, meta


df, meta = load_text_dataset(DATASET_NAME)
df.head(3)

In [ ]:
def compute_dataset_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Per-class statistics on the training split.
    """
    train = df[df["split"] == "train"].copy()
    train["num_tokens"] = train["text"].str.split().str.len()
    total = len(train)

    stats = (
        train.groupby("label_name")["num_tokens"]
        .agg(
            count="count",
            mean_length="mean",
            median_length="median",
            min_length="min",
            max_length="max",
        )
        .round(1)
    )
    stats["proportion"] = (stats["count"] / total).round(3)
    return stats[["count", "proportion", "mean_length", "median_length", "min_length", "max_length"]]


stats = compute_dataset_stats(df)
stats

In [ ]:
def plot_class_distribution(df: pd.DataFrame, split: str = "train") -> None:
    """
    Bar chart of class counts, sorted descending, with count labels.
    """
    subset = df[df["split"] == split]
    counts = subset["label_name"].value_counts()

    ax = plt.gca()
    bars = ax.bar(counts.index, counts.values, color="steelblue")
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + counts.max() * 0.01,
                f"{val:,}", ha="center", va="bottom", fontsize=9)
    ax.set_title(f"Class distribution — {split} ({len(subset):,} examples)")
    ax.set_xlabel("Class")
    ax.set_ylabel("Count")
    plt.xticks(rotation=20, ha="right")


def plot_length_distribution(df: pd.DataFrame, split: str = "train", max_tokens: int = 200) -> None:
    """
    Overlapping histograms of token counts, clipped at max_tokens.
    """
    subset = df[df["split"] == split].copy()
    subset["num_tokens"] = subset["text"].str.split().str.len().clip(upper=max_tokens)

    ax = plt.gca()
    for label in subset["label_name"].unique():
        lengths = subset[subset["label_name"] == label]["num_tokens"]
        ax.hist(lengths, bins=40, alpha=0.5, label=label)
    ax.set_xlabel("Token count (whitespace)")
    ax.set_ylabel("Frequency")
    ax.set_title(f"Text length distribution — {split} (clipped at {max_tokens})")
    ax.legend()


fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plt.sca(axes[0]); plot_class_distribution(df, split="train")
plt.sca(axes[1]); plot_length_distribution(df, split="train")
plt.tight_layout()

In [ ]:
def make_splits(
    df: pd.DataFrame,
    val_size: float = 0.1,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build train / val / test splits.
    Reuses existing splits where available; creates val from train otherwise.
    """
    available = df["split"].unique().tolist()

    test_df = df[df["split"] == "test"].copy() if "test" in available else None

    if "validation" in available:
        train_df = df[df["split"] == "train"].copy()
        val_df   = df[df["split"] == "validation"].copy()
    else:
        full_train = df[df["split"] == "train"].copy()
        train_df, val_df = train_test_split(
            full_train,
            test_size=val_size,
            stratify=full_train["label"],
            random_state=random_state,
        )

    # If no test split, carve from train
    if test_df is None:
        train_df, test_df = train_test_split(
            train_df,
            test_size=0.1,
            stratify=train_df["label"],
            random_state=random_state,
        )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


train_df, val_df, test_df = make_splits(df, val_size=0.1)
print(f"Train : {len(train_df):,}  |  Val : {len(val_df):,}  |  Test : {len(test_df):,}")
for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    props = split["label_name"].value_counts(normalize=True).round(3).to_dict()
    print(f"{name}: {props}")

---
## Part 4 — Majority-Class Baseline

In [ ]:
def majority_baseline(
    train_df: pd.DataFrame,
    eval_df: pd.DataFrame,
) -> tuple[list, list]:
    """
    Always predict the most frequent class in train_df.
    """
    majority_label = train_df["label"].mode()[0]
    y_true = eval_df["label"].tolist()
    y_pred = [majority_label] * len(y_true)
    return y_true, y_pred


y_true_val,  y_pred_val  = majority_baseline(train_df, val_df)
y_true_test, y_pred_test = majority_baseline(train_df, test_df)

print("=== Validation ===")
print(classification_report(y_true_val, y_pred_val, target_names=meta["label_names"], zero_division=0))
print("=== Test ===")
print(classification_report(y_true_test, y_pred_test, target_names=meta["label_names"], zero_division=0))

In [ ]:
def plot_confusion_matrix(
    y_true: list,
    y_pred: list,
    label_names: list[str],
    title: str = "Confusion Matrix",
    normalise: bool = True,
) -> None:
    """
    Plot a (optionally row-normalised) confusion matrix.
    """
    cm = confusion_matrix(y_true, y_pred)
    if normalise:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        fmt = ".2f"
    else:
        fmt = "d"

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt=fmt, cmap="Blues",
        xticklabels=label_names, yticklabels=label_names,
        vmin=0, vmax=1 if normalise else None, ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


plot_confusion_matrix(y_true_test, y_pred_test, meta["label_names"], title="Majority Baseline — Test")

---
## Part 5 — Error Analysis

In [ ]:
def find_errors(
    eval_df: pd.DataFrame,
    y_true: list,
    y_pred: list,
    label_names: list[str],
    n: int = 10,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Return a stratified sample of n misclassified examples.
    """
    result = eval_df.copy().reset_index(drop=True)
    result["y_true"] = y_true
    result["y_pred"] = y_pred

    errors = result[result["y_true"] != result["y_pred"]].copy()
    errors["true_label"] = errors["y_true"].apply(lambda i: label_names[i])
    errors["predicted"]  = errors["y_pred"].apply(lambda i: label_names[i])
    errors["num_tokens"] = errors["text"].str.split().str.len()

    # Stratified sample: take floor(n / num_classes) per true class
    n_per_class = max(1, n // errors["true_label"].nunique())
    sampled = (
        errors.groupby("true_label", group_keys=False)
        .apply(lambda g: g.sample(min(len(g), n_per_class), random_state=random_state))
    )
    return sampled[["text", "true_label", "predicted", "num_tokens"]].head(n)


errors = find_errors(test_df, y_true_test, y_pred_test, meta["label_names"], n=10)
pd.set_option("display.max_colwidth", 120)
errors

In [ ]:
def compute_vocab_overlap(train_df: pd.DataFrame, top_n: int = 500) -> pd.DataFrame:
    """
    Pairwise Jaccard similarity of top-N vocabulary sets per class.
    """
    class_vocabs = {}
    for label in train_df["label_name"].unique():
        texts  = train_df[train_df["label_name"] == label]["text"]
        tokens = [tok for text in texts for tok in preprocess_text(text).split()]
        top    = {tok for tok, _ in Counter(tokens).most_common(top_n)}
        class_vocabs[label] = top

    labels  = sorted(class_vocabs)
    matrix  = pd.DataFrame(index=labels, columns=labels, dtype=float)
    for a in labels:
        for b in labels:
            intersection = len(class_vocabs[a] & class_vocabs[b])
            union        = len(class_vocabs[a] | class_vocabs[b])
            matrix.loc[a, b] = round(intersection / union, 3) if union else 0.0
    return matrix


overlap = compute_vocab_overlap(train_df, top_n=500)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(overlap.astype(float), annot=True, fmt=".2f", cmap="Blues", ax=ax, vmin=0, vmax=1)
ax.set_title("Vocabulary Jaccard Overlap — top-500 tokens per class")
plt.tight_layout()
plt.show()

---
## Summary

In [ ]:
summary = pd.DataFrame({
    "Metric": ["Accuracy", "Macro F1", "Weighted F1"],
    "Majority Baseline (test)": [
        f"{accuracy_score(y_true_test, y_pred_test):.3f}",
        f"{f1_score(y_true_test, y_pred_test, average='macro',    zero_division=0):.3f}",
        f"{f1_score(y_true_test, y_pred_test, average='weighted', zero_division=0):.3f}",
    ],
})

print(f"Dataset : {DATASET_NAME}  |  Train {len(train_df):,}  Val {len(val_df):,}  Test {len(test_df):,}")
display(summary)
display(comparison)

---
## Instructor notes

**Expected results on AG News (default):**

| Metric | Value |
|--------|-------|
| Majority class | Sports (label 1) — 25% of training set |
| Accuracy (test) | ~0.25 |
| Macro F1 (test) | ~0.10 |
| Weighted F1 (test) | ~0.125 |

AG News is perfectly balanced (25% per class), so the majority baseline gets exactly 1/K accuracy. This is a good teaching moment: ask students what the confusion matrix looks like before they plot it.

**Common mistakes to watch for:**
- Using `accuracy` instead of `f1_macro` as the headline metric for imbalanced classes
- Not resetting the index after `train_test_split` — causes silent alignment bugs in `find_errors`
- Applying `preprocess_text` before BPE — this loses casing and punctuation that BPE relies on

**Timing guide:**
- Part 1: ~20 min
- Part 2: ~25 min
- Part 3: ~20 min
- Part 4: ~20 min
- Part 5: ~20 min + 5 min summary review